# Comprehensive QLoRA Training Tutorial

This notebook is a hands-on guide to **QLoRA** (Quantized Low-Rank Adaptation)
fine-tuning with the [`training_hub`](https://github.com/Red-Hat-AI-Innovation-Team/training_hub)
library on Alauda AI.

QLoRA freezes the base model in **4-bit NormalFloat (NF4)** precision and trains a
small set of **LoRA adapter** matrices on top. Only the adapters are updated, so a
7B model that needs ~60 GiB for full SFT fits comfortably on a single 16-24 GiB GPU
slice — the whole point of QLoRA (Dettmers et al., 2023, [arXiv:2305.14314](https://arxiv.org/abs/2305.14314)).

`training_hub` exposes this through a single `lora_sft(...)` call: set
`load_in_4bit=True` plus the LoRA rank/alpha and you have QLoRA.

> Runtime: run this notebook on the `traininghub0.1-cu126-amd64` runtime image, which
> bundles `trl`, `peft`, and **`bitsandbytes`** — the 4-bit quantization backend QLoRA needs.

## When to use QLoRA

| | Full SFT | LoRA | **QLoRA** |
|---|---|---|---|
| Base weights | trainable (bf16) | frozen (bf16) | **frozen (4-bit NF4)** |
| Trainable params | 100% | adapters only | adapters only |
| GPU memory | highest | medium | **lowest** |
| Output | full checkpoint | base + adapter | base + adapter |

Reach for QLoRA when you are GPU-memory bound — a large model on a single card or a
small HAMi vGPU slice. The trade-off is a small quantization-induced quality gap and
slightly slower steps (de-quantization happens on the fly). For maximum throughput when
memory is not the constraint, use plain `sft(...)` / `lora_sft(..., load_in_4bit=False)`.

In [ ]:
import json
import os

# QLoRA trains on the same chat-style JSONL schema as SFT.
data_path = "./test_qlora_data.jsonl"
if not os.path.exists(data_path):
    print(f"Creating dummy dataset at {data_path}")
    dummy_data = [
        {"messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Hello, how are you?"},
            {"role": "assistant", "content": "I am doing well, thank you! How can I help you today?"},
        ]}
    ] * 10
    with open(data_path, "w") as f:
        for d in dummy_data:
            f.write(json.dumps(d) + "\n")

## Data format requirements

QLoRA uses the **same conversational JSONL** as SFT — one conversation per line:

```json
{"messages": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

Only `assistant` turns contribute to the loss by default. See the
[SFT data format](../training-hub-fine-tuning.mdx#data-format) for the full masking rules.

## Model configuration examples

QLoRA shines on larger models, but you can adapt any HuggingFace causal-LM checkpoint.
Provide a HuggingFace name or a local path. For air-gapped clusters, pre-download the
model (e.g. from a ModelScope mirror) to a local directory and point `model_path` at it.

In [ ]:
# =============================================================================
# MODEL CONFIGURATION EXAMPLES
# Pick one as a starting point and tune for your hardware.
# =============================================================================

# Example 1: Qwen 3 0.6B (tiny - good for a single small GPU slice / smoke test)
qwen_small_example = {
    "model_name": "Qwen 3 0.6B",
    "model_path": "/opt/app-root/src/Qwen3-0.6B",  # local path or HF name
    "lora_r": 8, "lora_alpha": 16,
    "example_max_seq_len": 2048,
    "example_batch_size": 8,
    "example_learning_rate": 2e-4,
}

# Example 2: Qwen 2.5 7B Instruct - the canonical QLoRA target
qwen_7b_example = {
    "model_name": "Qwen 2.5 7B Instruct",
    "model_path": "Qwen/Qwen2.5-7B-Instruct",
    "lora_r": 16, "lora_alpha": 32,
    "example_max_seq_len": 4096,
    "example_batch_size": 16,
    "example_learning_rate": 2e-4,
}

# Example 3: Llama 3.1 8B Instruct
llama_8b_example = {
    "model_name": "Llama 3.1 8B Instruct",
    "model_path": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "lora_r": 16, "lora_alpha": 32,
    "example_max_seq_len": 4096,
    "example_batch_size": 16,
    "example_learning_rate": 1e-4,
}

# =============================================================================
# SELECT YOUR CONFIGURATION
# =============================================================================
selected_example = qwen_small_example
print(f"Selected model: {selected_example['model_name']} ({selected_example['model_path']})")

## QLoRA & quantization parameters

`lora_sft(...)` accepts the LoRA hyper-parameters plus the bitsandbytes quantization
knobs that turn LoRA into QLoRA:

- `load_in_4bit=True` — load the frozen base model in 4-bit (this is what makes it QLoRA).
- `bnb_4bit_quant_type="nf4"` — NormalFloat-4 quantization (recommended over `"fp4"`).
- `bnb_4bit_compute_dtype="bfloat16"` — de-quantize to bf16 for the matmuls.
- `bnb_4bit_use_double_quant=True` — nested quantization of the quant constants (extra memory saving).
- `lora_r` / `lora_alpha` / `lora_dropout` / `target_modules` — the LoRA adapter shape.

In [ ]:
# =============================================================================
# COMPLETE QLoRA PARAMETER CONFIGURATION
# =============================================================================
from datetime import datetime

experiment_name = "qlora_comprehensive_example"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
full_experiment_name = f"{experiment_name}_{timestamp}"

# ---- Required ----
model_path = selected_example["model_path"]
data_path = "./test_qlora_data.jsonl"
ckpt_output_dir = f"checkpoints/{full_experiment_name}"

# ---- LoRA adapter shape ----
lora_r = selected_example["lora_r"]            # rank of the low-rank update
lora_alpha = selected_example["lora_alpha"]    # scaling (commonly 2x rank)
lora_dropout = 0.05
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]  # attention projections; "all-linear" also works

# ---- 4-bit quantization (this is what makes it QLoRA) ----
load_in_4bit = True
bnb_4bit_quant_type = "nf4"
bnb_4bit_compute_dtype = "bfloat16"
bnb_4bit_use_double_quant = True

# ---- Core training ----
num_epochs = 3
effective_batch_size = selected_example["example_batch_size"]
learning_rate = selected_example["example_learning_rate"]   # LoRA tolerates higher LRs than full SFT
max_seq_len = selected_example["example_max_seq_len"]
warmup_steps = 10

print("QLoRA configuration:")
print(f"  4-bit: load_in_4bit={load_in_4bit}, quant_type={bnb_4bit_quant_type}, "
      f"compute_dtype={bnb_4bit_compute_dtype}, double_quant={bnb_4bit_use_double_quant}")
print(f"  LoRA:  r={lora_r}, alpha={lora_alpha}, dropout={lora_dropout}, targets={target_modules}")
print(f"  Train: epochs={num_epochs}, ebs={effective_batch_size}, lr={learning_rate}, max_seq_len={max_seq_len}")

## Distributed configuration

QLoRA is single-GPU friendly — that is its main attraction — but `lora_sft` accepts the
same distributed topology knobs as `sft`. Keep `nproc_per_node=1` for a single GPU.

In [ ]:
# Single-GPU is the common QLoRA case.
nproc_per_node = 1
nnodes = 1
node_rank = 0
rdzv_id = 100
rdzv_endpoint = "127.0.0.1:29500"
print(f"distributed: nproc_per_node={nproc_per_node}, nnodes={nnodes}")

## Execute training

The final cell calls `training_hub.lora_sft(...)`. With `load_in_4bit=True` this performs
QLoRA: the base model is held in 4-bit NF4 and only the LoRA adapters are trained.

In [ ]:
from training_hub import lora_sft

result = lora_sft(
    # --- required ---
    model_path=model_path,
    data_path=data_path,
    ckpt_output_dir=ckpt_output_dir,
    # --- LoRA adapter ---
    lora_r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=target_modules,
    # --- 4-bit QLoRA quantization ---
    load_in_4bit=load_in_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=bnb_4bit_compute_dtype,
    bnb_4bit_use_double_quant=bnb_4bit_use_double_quant,
    # --- core training ---
    num_epochs=num_epochs,
    effective_batch_size=effective_batch_size,
    learning_rate=learning_rate,
    max_seq_len=max_seq_len,
    warmup_steps=warmup_steps,
    # --- distributed ---
    nproc_per_node=nproc_per_node,
    nnodes=nnodes,
    node_rank=node_rank,
    rdzv_id=rdzv_id,
    rdzv_endpoint=rdzv_endpoint,
)
print(f"QLoRA training finished: {result!r}")

## Post-training analysis

QLoRA writes a **LoRA adapter** (a few MB) rather than a full checkpoint. To serve the
model you either (a) load the base model + adapter together, or (b) merge the adapter into
the base weights once and export a standalone checkpoint.

In [ ]:
import os

print(f"Checkpoint dir: {ckpt_output_dir}")
if os.path.isdir(ckpt_output_dir):
    for root, _dirs, files in os.walk(ckpt_output_dir):
        for fn in files:
            if fn.endswith((".safetensors", ".bin", ".json")):
                print("  ", os.path.relpath(os.path.join(root, fn), ckpt_output_dir))

# Merge the adapter into the base model for standalone inference:
#   from peft import AutoPeftModelForCausalLM
#   m = AutoPeftModelForCausalLM.from_pretrained("<adapter_dir>")
#   m = m.merge_and_unload()
#   m.save_pretrained("merged-model")
print("\nTo serve: load base + adapter with peft, or merge_and_unload() for a standalone model.")

## Parameter reference summary

### QLoRA-specific parameters

| Parameter | Description | Typical |
|---|---|---|
| `load_in_4bit` | Enable 4-bit base — turns LoRA into QLoRA | `True` |
| `bnb_4bit_quant_type` | 4-bit data type | `"nf4"` |
| `bnb_4bit_compute_dtype` | Matmul compute dtype | `"bfloat16"` |
| `bnb_4bit_use_double_quant` | Nested quantization | `True` |
| `lora_r` | LoRA rank | 8-64 |
| `lora_alpha` | LoRA scaling | 2x `lora_r` |
| `lora_dropout` | Adapter dropout | 0.0-0.1 |
| `target_modules` | Layers to adapt | attention proj / `"all-linear"` |
| `load_in_8bit` | 8-bit base (LoRA-8bit, not QLoRA) | `False` |

### Common training parameters

| Parameter | Description |
|---|---|
| `model_path`, `data_path`, `ckpt_output_dir` | Required I/O |
| `num_epochs`, `effective_batch_size`, `learning_rate`, `max_seq_len` | Core training |
| `warmup_steps`, `lr_scheduler` | LR schedule |
| `nproc_per_node`, `nnodes`, `node_rank`, `rdzv_id`, `rdzv_endpoint` | Distributed topology |

> **Hardware note:** 4-bit QLoRA via bitsandbytes requires an NVIDIA GPU of compute
> capability **sm_75 or newer** (Turing/Ampere/Hopper). Older cards (e.g. P100, sm_60)
> are not supported by the bitsandbytes 4-bit kernels.
>
> **NPU note:** On Huawei Ascend NPU, bitsandbytes 4-bit is not available. Use LoRA
> (without 4-bit) on the `llamafactory0.9-cann8.5-arm64` runtime
> (`finetuning_type: lora`) as the parameter-efficient path — see
> [Fine-tune and Pretrain on Ascend NPU](../fine-tune-and-pretrain-llms-on-ascend-npu.mdx).